# Subsample JEPAHypothesis: the T-Net should map **different numerical observations of the samefunction** to similar latents.```textsame equation f  ├── view 0 -> T-Net -> z_0 -> symbolic decoder -> CE loss  └── view 1 -> T-Net -> z_1                 subsample_loss = mean over pairs of (1 - cos(z_i, z_j))                 total = CE + lambda * subsample_loss```No symbolic-expression target, no MLP predictor, no `[PRED]` tokens.Separate checkpoints from `jepa_sweep`.

In [ ]:
# Environment setup — works on both Colab and localimport os, sysIN_COLAB = 'google.colab' in sys.modulesif IN_COLAB:    from google.colab import drive    drive.mount('/content/drive')    REPO_DIR = '/content/drive/MyDrive/Symba/symbolic-jepa'    if not os.path.exists(REPO_DIR):        %cd /content/drive/MyDrive/Symba        !git clone https://github.com/zzpDavid2/symbolic-jepa.git {REPO_DIR}    os.chdir(REPO_DIR)    sys.path.insert(0, REPO_DIR)    !pip install -q sympy scipy    CKPT_DIR = '/content/drive/MyDrive/Symba/symbolic-jepa/checkpoints/subsample_jepa'    LOG_DIR  = '/content/drive/MyDrive/Symba/symbolic-jepa/runs_subsample'else:    CKPT_DIR = 'checkpoints/subsample_jepa'    LOG_DIR  = 'runs_subsample'os.makedirs(CKPT_DIR, exist_ok=True)os.makedirs(LOG_DIR, exist_ok=True)print(f'Environment: {"Colab" if IN_COLAB else "Local"}')print(f'Checkpoints: {CKPT_DIR}')

In [ ]:
if IN_COLAB:    %cd /content/drive/MyDrive/Symba/symbolic-jepa    !git pull

In [ ]:
import gcimport jsonimport randomimport timeimport numpy as npimport torchfrom torch.utils.data import DataLoaderfrom tqdm.auto import tqdmfrom symbolic_jepa import (    PrefixTokenizer, Expression,    TNet, SymbolicTransformer,    subsample_consistency_loss,    PointCloudDataset, MultiViewPointCloudDataset,    build_multiview_synthetic_splits, load_synthetic_pkl,    teacher_forced_counts, evaluate_predictions,    view_consistency_diagnostics, generate_diagnostic_embeddings,)from symbolic_jepa.tokenizer import prefix_to_sympytorch.backends.cudnn.deterministic = Truetorch.backends.cudnn.benchmark = Falseif torch.cuda.is_available():    DEVICE = 'cuda'elif torch.backends.mps.is_available():    DEVICE = 'mps'else:    DEVICE = 'cpu'print(f'Device: {DEVICE}')

In [ ]:
# ── Model / training hyperparameters (unchanged from jepa_sweep) ──MAX_VARS    = 1               # univariate synthetic dataD_INPUT     = MAX_VARS + 1    # (x, y) = 2N_POINTS    = 1000MAX_SEQ     = 64D_MODEL     = 512N_HEADS     = 8N_LAYERS    = 4DROPOUT     = 0.2EPOCHS      = 30LR          = 3e-4BATCH       = 16VAL_EVERY   = 1USE_AMP     = True# ── Subsample-JEPA config ──N_VIEWS          = 2       # views per equation during trainingTRAIN_VIEW_SEED  = 1729    # governs training views (independent of model seed)EVAL_SEED        = 2718    # governs fixed diagnostic viewsDIAG_N_VIEWS     = 10      # views per equation for the consistency diagnosticDIAG_N_EXPRS     = 50      # val equations used for the diagnostic# Which cosine the LOSS optimises (the diagnostics always report both).#   'centered' — subtract each view's batch mean first.  RECOMMENDED.#   'cosine'   — ordinary pairwise cosine on the raw embeddings.# The T-Net max-pools ReLU features into the positive orthant, so raw cosine# between any two embeddings sits near 0.99 and the raw loss starts ~5e-4:# too small to produce gradient. In a d_model=128 A/B the raw objective at# lambda=0.3 left val loss identical to baseline to 4 decimals and moved# same-function cosine by -0.002 (noise), while centered moved it +0.031 and# cut its own objective 4.5x. Raw is also *minimised* by collapse; centered# penalises it. Re-run the A/B cell below to confirm at full scale.SUBSAMPLE_LOSS = 'centered'# ── Sweep config ──LAMBDA_VALUES = [0, 0.003, 0.01, 0.03, 0.1, 0.3]SEEDS         = [42, 123, 7]VERSION_TAG   = f'subsample_{SUBSAMPLE_LOSS}_v1'# Synthetic data (pre-generated by SYMBA_Reg_Data_Gen notebook)SYNTH_PKL   = 'data/synthetic.pkl'MAX_SYNTH   = 10_000SYNTH_SEED  = 42

## Load synthetic data

In [ ]:
tokenizer = PrefixTokenizer(max_vars=MAX_VARS)print(f'Vocab size: {len(tokenizer)}')print(f'Loading synthetic expressions from {SYNTH_PKL}...')synth_exprs = load_synthetic_pkl(    SYNTH_PKL, max_seq_len=MAX_SEQ,    tokenizer=tokenizer, max_expressions=MAX_SYNTH,)print(f'Loaded {len(synth_exprs)} expressions')

In [ ]:
# Train is multi-view; val/test stay deterministic single-view.# Same shuffle/seed as build_synthetic_splits, so the partition matches# the jepa_sweep runs and results are comparable.synth_train, synth_val, synth_test = build_multiview_synthetic_splits(    synth_exprs, tokenizer,    n_points=N_POINTS, max_seq_len=MAX_SEQ, max_vars=MAX_VARS,    seed=SYNTH_SEED, n_views=N_VIEWS, train_view_seed=TRAIN_VIEW_SEED,)

## Training / evaluation functions

In [ ]:
from torch.utils.tensorboard import SummaryWriter

In [ ]:
def seed_everything(seed):    """Full re-seed for model init and training stochasticity."""    torch.manual_seed(seed)    np.random.seed(seed)    random.seed(seed)    if torch.cuda.is_available():        torch.cuda.manual_seed_all(seed)def seed_worker(worker_id):    worker_seed = torch.initial_seed() % 2**32    np.random.seed(worker_seed + worker_id)    random.seed(worker_seed + worker_id)def build_model(tokenizer, dropout=DROPOUT):    encoder = TNet(d_input=D_INPUT, d_model=D_MODEL)    model = SymbolicTransformer(        encoder=encoder, vocab_size=len(tokenizer),        d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,        d_ff=4 * D_MODEL, max_seq_len=MAX_SEQ,        dropout=dropout, pad_id=tokenizer.pad_id,    ).to(DEVICE)    return model, encoderdef amp_context():    if USE_AMP and DEVICE == 'cuda':        return lambda: torch.autocast('cuda', dtype=torch.bfloat16)    if USE_AMP and DEVICE == 'mps':        return lambda: torch.autocast('mps', dtype=torch.float16)    return lambda: torch.amp.autocast('cpu', enabled=False)def view_consistency(model, val_ds):    """same/diff-function cosine on fixed diagnostic views."""    z = generate_diagnostic_embeddings(        val_ds, model.encoder,        n_exprs=DIAG_N_EXPRS, n_views=DIAG_N_VIEWS,        eval_seed=EVAL_SEED, device=DEVICE, batch_size=BATCH,    )    return view_consistency_diagnostics(z, DIAG_N_VIEWS)

In [ ]:
def train_one(lam, seed, synth_train, synth_val, tokenizer,              epochs=EPOCHS, tag=VERSION_TAG):    """Train one (lambda, seed) run. Training only — no symbolic eval."""    seed_everything(seed)    run_tag = f'lam{lam}_seed{seed}'    run_dir = f'{CKPT_DIR}/{tag}/{run_tag}'    CKPT_PATH = f'{run_dir}/latest.pt'    BEST_PATH = f'{run_dir}/best.pt'    if os.path.exists(CKPT_PATH):        ck = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)        if ck['epoch'] >= epochs:            print(f'{run_tag}: training complete (epoch {ck["epoch"]}), SKIPPING')            return    os.makedirs(run_dir, exist_ok=True)    print(f'\n{"="*60}')    print(f'{run_tag} | n_views={N_VIEWS} | loss={SUBSAMPLE_LOSS} | '          f'view_seed={TRAIN_VIEW_SEED} | tag={tag}')    print(f'{"="*60}')    model, encoder = build_model(tokenizer)    g = torch.Generator()    g.manual_seed(seed)    # persistent_workers=False is REQUIRED: workers fork a copy of the dataset,    # so persistent workers would never see `synth_train.epoch` updates and the    # views would silently freeze at epoch 0.    train_loader = DataLoader(synth_train, batch_size=BATCH, shuffle=True,                              num_workers=2, persistent_workers=False,                              pin_memory=True, worker_init_fn=seed_worker,                              generator=g)    val_loader = DataLoader(synth_val, batch_size=BATCH, shuffle=False,                            num_workers=2, persistent_workers=True,                            worker_init_fn=seed_worker)    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)    start_epoch = 1    best_val = float('inf')    best_val_acc = 0.0    best_vc = {}    history = {'train': [], 'val': []}    if os.path.exists(CKPT_PATH):        ck = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)        model.load_state_dict(ck['model'])        optimizer.load_state_dict(ck['optimizer'])        scheduler.load_state_dict(ck['scheduler'])        start_epoch = ck['epoch'] + 1        best_val = ck['best_val']        best_val_acc = ck.get('best_val_acc', 0.0)        best_vc = ck.get('best_view_consistency', {})        history = ck['history']        print(f'  Resuming from epoch {start_epoch} (best val: {best_val:.4f})')    amp_ctx = amp_context()    writer = SummaryWriter(log_dir=f'{LOG_DIR}/{tag}/{run_tag}')    for epoch in range(start_epoch, epochs + 1):        # Fresh, reproducible views for this epoch.        synth_train.epoch = epoch        model.train()        train_loss_gen = 0        train_loss_sub = 0        train_sub_other = 0   # the mode we are NOT optimising, for reference        pbar = tqdm(train_loader, desc=f'{run_tag} E{epoch}/{epochs}', leave=False)        for batch in pbar:            points_views = batch['points_views'].to(DEVICE, non_blocking=True)            input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)            attn_mask = batch['attn_mask'].to(DEVICE, non_blocking=True)            optimizer.zero_grad()            with amp_ctx():                # View 0 drives the ordinary symbolic-regression forward pass.                out = model(points_views[:, 0], input_ids, attn_mask=attn_mask)                loss_gen = out['loss']                if lam > 0:                    # z for view 0 is already computed; encode the rest.                    z_views = [out['z_num']] + [                        model.encoder(points_views[:, v])                        for v in range(1, N_VIEWS)                    ]                    loss_sub = subsample_consistency_loss(                        z_views, mode=SUBSAMPLE_LOSS)                    loss = loss_gen + lam * loss_sub                    # Track the other mode too — free, and shows how the                    # two scales move relative to each other.                    with torch.no_grad():                        other = 'cosine' if SUBSAMPLE_LOSS == 'centered' else 'centered'                        loss_sub_other = subsample_consistency_loss(                            z_views, mode=other)                else:                    loss_sub = loss_sub_other = None                    loss = loss_gen            loss.backward()            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)            optimizer.step()            train_loss_gen += loss_gen.item()            postfix = {'ce': f'{loss_gen.item():.4f}'}            global_step = (epoch - 1) * len(train_loader) + pbar.n            writer.add_scalar('train/loss_step', loss_gen.item(), global_step)            if loss_sub is not None:                train_loss_sub += loss_sub.item()                train_sub_other += loss_sub_other.item()                postfix['sub'] = f'{loss_sub.item():.4f}'                writer.add_scalar('train/subsample', loss_sub.item(), global_step)                writer.add_scalar('train/subsample_other',                                  loss_sub_other.item(), global_step)                writer.add_scalar('train/subsample_weighted',                                  (lam * loss_sub).item(), global_step)            pbar.set_postfix(postfix)        scheduler.step()        train_avg = train_loss_gen / len(train_loader)        history['train'].append(train_avg)        history.setdefault('train_sub', []).append(train_loss_sub / len(train_loader))        history.setdefault('train_sub_other', []).append(            train_sub_other / len(train_loader))        writer.add_scalar('train/loss_epoch', train_avg, epoch)        writer.add_scalar('train/lr', scheduler.get_last_lr()[0], epoch)        # ── Validation (token-weighted aggregation) ──        if epoch % VAL_EVERY == 0 or epoch == epochs:            model.eval()            val_loss_sum = 0.0            val_tokens_total = 0            acc_correct = 0.0            acc_total = 0.0            with torch.no_grad(), amp_ctx():                for batch in val_loader:                    points    = batch['points'].to(DEVICE, non_blocking=True)                    input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)                    attn_mask = batch['attn_mask'].to(DEVICE, non_blocking=True)                    out = model(points, input_ids, attn_mask=attn_mask)                    n_tok = out['n_tokens']                    val_loss_sum += out['loss'].item() * n_tok                    val_tokens_total += n_tok                    c, t = teacher_forced_counts(out['logits'], input_ids,                                                 tokenizer.pad_id)                    acc_correct += c                    acc_total += t            val_avg = val_loss_sum / max(val_tokens_total, 1)            val_acc_avg = acc_correct / max(acc_total, 1)            history['val'].append(val_avg)            history.setdefault('val_acc', []).append(val_acc_avg)            writer.add_scalar('val/loss', val_avg, epoch)            writer.add_scalar('val/token_accuracy', val_acc_avg, epoch)            # Representation consistency on fixed diagnostic views            vc = view_consistency(model, synth_val)            for k, v in vc.items():                writer.add_scalar(f'val/{k}', v, epoch)                history.setdefault(f'val_{k}', []).append(v)            is_best = val_avg < best_val            if is_best:                best_val = val_avg                best_val_acc = val_acc_avg                best_vc = vc            flag = ' * best' if is_best else ''            print(f'  E{epoch}/{epochs} | train={train_avg:.4f} | '                  f'val={val_avg:.4f}{flag} | acc={val_acc_avg*100:.1f}% | '                  f'gap={vc["gap"]:.4f} gap_c={vc["gap_centered"]:.4f}')            if is_best:                torch.save({'model': model.state_dict(), 'epoch': epoch,                            'val': val_avg, 'val_acc': val_acc_avg,                            'view_consistency': vc}, BEST_PATH)        torch.save({            'model': model.state_dict(),            'optimizer': optimizer.state_dict(),            'scheduler': scheduler.state_dict(),            'epoch': epoch,            'best_val': best_val,            'best_val_acc': best_val_acc,            'best_view_consistency': best_vc,            'history': history,            'lambda_subsample': lam,            'subsample_loss': SUBSAMPLE_LOSS,            'seed': seed,            'n_views': N_VIEWS,            'train_view_seed': TRAIN_VIEW_SEED,        }, CKPT_PATH)    writer.close()    del train_loader, val_loader, model, encoder, optimizer, scheduler    if DEVICE == 'cuda':        torch.cuda.empty_cache()    gc.collect()

In [ ]:
def eval_one(lam, seed, synth_val, synth_test, tokenizer, tag=VERSION_TAG):    """Load the best checkpoint and evaluate on the full test set."""    run_tag = f'lam{lam}_seed{seed}'    run_dir = f'{CKPT_DIR}/{tag}/{run_tag}'    metrics_path = f'{run_dir}/metrics.json'    BEST_PATH = f'{run_dir}/best.pt'    CKPT_PATH = f'{run_dir}/latest.pt'    if os.path.exists(metrics_path):        with open(metrics_path) as f:            return json.load(f)    model, encoder = build_model(tokenizer, dropout=0.0)    ck_path = BEST_PATH if os.path.exists(BEST_PATH) else CKPT_PATH    ck = torch.load(ck_path, map_location=DEVICE, weights_only=False)    model.load_state_dict(ck['model'])    model.eval()    latest_ck = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)    best_val = ck.get('val', latest_ck.get('best_val', float('inf')))    best_val_acc = ck.get('val_acc', latest_ck.get('best_val_acc', 0.0))    vc = ck.get('view_consistency') or latest_ck.get('best_view_consistency') or {}    if not vc:        vc = view_consistency(model, synth_val)    # Greedy decode over the FULL test set    eval_loader = DataLoader(synth_test, batch_size=BATCH, shuffle=False)    greedy_preds = []    for batch in tqdm(eval_loader, desc=f'{run_tag} decode', leave=False):        points = batch['points'].to(DEVICE)        input_ids = batch['input_ids']        preds = model.generate(points, tokenizer, max_new_tokens=MAX_SEQ)        for j, pred_str in enumerate(preds):            greedy_preds.append((tokenizer.decode(input_ids[j].tolist()), pred_str))    res = evaluate_predictions(greedy_preds, synth_test, tokenizer)    metrics = {        'lambda': lam,        'seed': seed,        'run_tag': run_tag,        'version_tag': tag,        'n_views': N_VIEWS,        'train_view_seed': TRAIN_VIEW_SEED,        'subsample_loss': SUBSAMPLE_LOSS,        'best_val_loss': best_val,        'best_val_acc': best_val_acc,        'same_fn_cos': vc.get('same_fn_cos', float('nan')),        'diff_fn_cos': vc.get('diff_fn_cos', float('nan')),        'cos_gap': vc.get('gap', float('nan')),        'same_fn_cos_centered': vc.get('same_fn_cos_centered', float('nan')),        'diff_fn_cos_centered': vc.get('diff_fn_cos_centered', float('nan')),        'cos_gap_centered': vc.get('gap_centered', float('nan')),        'greedy_exact_match': res['exact_match'],        'greedy_token_acc': res['token_accuracy'],        'greedy_algebraic_equiv': res['algebraic_equiv'],        'greedy_r2_above_0.9': res['r2_above_0.9'],        'mean_r2': res['mean_r2'],        'median_r2': res['median_r2'],        'n_parseable': res['n_parseable'],        'n_total': res['n_total'],        'details': res['details'],    }    with open(metrics_path, 'w') as f:        json.dump(metrics, f, indent=2)    del model, encoder, eval_loader    if DEVICE == 'cuda':        torch.cuda.empty_cache()    gc.collect()    return metrics

## Smoke testRun this **before** the sweep. Verifies the paired-view data path, the lossterms, gradient flow, and that `lambda=0` still behaves like a plain baseline.

In [ ]:
# ── Static checks on the data path and one training step ──_ds = synth_train_ds.epoch = 0_item = _ds[0]_pv = _item['points_views']print('points_views shape:', tuple(_pv.shape),      f'(expect ({N_VIEWS}, {N_POINTS}, {D_INPUT}))')assert _pv.shape == (N_VIEWS, N_POINTS, D_INPUT)_d = (_pv[0] - _pv[1]).abs().max().item()print(f'max |view0 - view1| = {_d:.4f}   (must be > 0: independent samples)')assert _d > 0, 'views identical — resampling is broken'# Same epoch -> identical views; next epoch -> different viewsassert torch.equal(_ds[0]['points_views'], _pv), 'views not reproducible'_ds.epoch = 1assert not torch.allclose(_ds[0]['points_views'], _pv), 'views frozen across epochs'_ds.epoch = 0print('views: reproducible within an epoch, refreshed across epochs  OK')# One forward/backward passseed_everything(42)_model, _enc = build_model(tokenizer)_loader = DataLoader(_ds, batch_size=BATCH, shuffle=False)_batch = next(iter(_loader))_pvb = _batch['points_views'].to(DEVICE)_ids = _batch['input_ids'].to(DEVICE)_mask = _batch['attn_mask'].to(DEVICE)_out = _model(_pvb[:, 0], _ids, attn_mask=_mask)_zs = [_out['z_num']] + [_model.encoder(_pvb[:, v]) for v in range(1, N_VIEWS)]_ce = _out['loss']_sub = subsample_consistency_loss(_zs)print(f'\nCE loss       = {_ce.item():.4f}  finite={torch.isfinite(_ce).item()}')print(f'subsample loss = {_sub.item():.4f}  finite={torch.isfinite(_sub).item()}')assert torch.isfinite(_ce) and torch.isfinite(_sub)(_ce + 0.03 * _sub).backward()_g = [p.grad for p in _enc.parameters() if p.grad is not None]print(f'encoder tensors with gradient: {len(_g)}, all finite: '      f'{all(torch.isfinite(x).all().item() for x in _g)}')assert _g and all(torch.isfinite(x).all() for x in _g)# Gradient must come from the subsample term too, not CE alone_model.zero_grad()subsample_consistency_loss(    [_model.encoder(_pvb[:, v]) for v in range(N_VIEWS)]).backward()_gs = sum(p.grad.abs().sum().item() for p in _enc.parameters() if p.grad is not None)print(f'encoder grad magnitude from subsample loss alone: {_gs:.4f}')assert _gs > 0, 'subsample loss does not reach the encoder'# Baseline view-consistency reading on the untrained modelprint('\nuntrained view consistency:', view_consistency(_model, synth_val))del _model, _enc, _loader, _batch, _pvb, _ids, _mask, _out, _zsgc.collect()print('\nStatic smoke checks PASSED')

In [ ]:
# ── Short training runs: lambda=0.03 and the lambda=0 baseline ──SMOKE_EPOCHS = 3SMOKE_TAG = 'smoke'for _lam in [0.03, 0]:    train_one(_lam, 42, synth_train, synth_val, tokenizer,              epochs=SMOKE_EPOCHS, tag=SMOKE_TAG)

In [ ]:
# ── Did same-function cosine actually rise, and did lambda=0 stay sane? ──for _lam in [0.03, 0]:    _p = f'{CKPT_DIR}/{SMOKE_TAG}/lam{_lam}_seed42/latest.pt'    if not os.path.exists(_p):        print(f'lambda={_lam}: no checkpoint'); continue    _h = torch.load(_p, map_location='cpu', weights_only=False)['history']    _same = _h.get('val_same_fn_cos_centered', [])    _gap = _h.get('val_gap_centered', [])    print(f'lambda={_lam}')    print(f'  val loss     : {[round(v, 4) for v in _h["val"]]}')    print(f'  subsample    : {[round(v, 4) for v in _h.get("train_sub", [])]}')    print(f'  same_c       : {[round(v, 4) for v in _same]}')    print(f'  gap_c        : {[round(v, 4) for v in _gap]}')    if len(_same) >= 2:        print(f'  same_c change: {_same[-1] - _same[0]:+.4f}')print('\nExpect: lambda=0.03 raises same_c more than lambda=0, gap_c stays')print('positive (no collapse), and val loss is comparable between the two.')print('lambda=0 must show subsample == 0.0 (extra encoder path skipped).')

## Loss-mode A/B: raw vs centered cosine`SUBSAMPLE_LOSS` defaults to `'centered'` on the strength of a small-scaleA/B (d_model=128): the raw objective at lambda=0.3 left validation loss**identical to baseline** and moved same-function cosine by -0.002 (noise),while centered moved it **+0.031** and cut its own objective 4.5x.This cell repeats that comparison at full model scale. It trains 3 short runs(baseline / raw / centered) and reports both loss values at every epoch, soyou can see which objective actually moves the representation and how the twoscales relate. Budget roughly 3 x `AB_EPOCHS` epochs of GPU time.

In [ ]:
# ── raw vs centered: which objective actually trains? ──AB_EPOCHS = 4AB_LAMBDA = 0.3AB_SEED   = 42def _ab_run(mode, lam, epochs=AB_EPOCHS, seed=AB_SEED):    """Short run measuring BOTH subsample losses; optimises only `mode`."""    seed_everything(seed)    model, encoder = build_model(tokenizer)    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)    g = torch.Generator(); g.manual_seed(seed)    tl = DataLoader(synth_train, batch_size=BATCH, shuffle=True,                    num_workers=2, persistent_workers=False,                    pin_memory=True, worker_init_fn=seed_worker, generator=g)    vl = DataLoader(synth_val, batch_size=BATCH, shuffle=False, num_workers=2)    amp_ctx = amp_context()    rows = []    for epoch in range(1, epochs + 1):        synth_train.epoch = epoch        model.train()        ce_s = raw_s = cen_s = n = 0        for b in tqdm(tl, desc=f'{mode} l={lam} E{epoch}', leave=False):            pv = b['points_views'].to(DEVICE, non_blocking=True)            ids = b['input_ids'].to(DEVICE, non_blocking=True)            msk = b['attn_mask'].to(DEVICE, non_blocking=True)            opt.zero_grad()            with amp_ctx():                out = model(pv[:, 0], ids, attn_mask=msk)                zs = [out['z_num']] + [model.encoder(pv[:, v])                                       for v in range(1, N_VIEWS)]                l_raw = subsample_consistency_loss(zs, mode='cosine')                l_cen = subsample_consistency_loss(zs, mode='centered')                active = l_raw if mode == 'cosine' else l_cen                total = out['loss'] + lam * active if lam > 0 else out['loss']            total.backward()            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)            opt.step()            ce_s += out['loss'].item(); raw_s += l_raw.item()            cen_s += l_cen.item(); n += 1        model.eval()        vls = vtt = ac = at = 0        with torch.no_grad(), amp_ctx():            for b in vl:                out = model(b['points'].to(DEVICE),                            b['input_ids'].to(DEVICE),                            attn_mask=b['attn_mask'].to(DEVICE))                k = out['n_tokens']; vls += out['loss'].item() * k; vtt += k                c, t = teacher_forced_counts(out['logits'],                                             b['input_ids'].to(DEVICE),                                             tokenizer.pad_id)                ac += c; at += t        vc = view_consistency(model, synth_val)        rows.append(dict(epoch=epoch, ce=ce_s / n, raw=raw_s / n, cen=cen_s / n,                         val=vls / max(vtt, 1), acc=ac / max(at, 1),                         same_c=vc['same_fn_cos_centered'],                         gap_c=vc['gap_centered']))        r = rows[-1]        print(f'  E{epoch} | CE={r["ce"]:.4f} sub_raw={r["raw"]:.6f} '              f'sub_cen={r["cen"]:.6f} | val={r["val"]:.4f} '              f'acc={r["acc"]*100:.1f}% same_c={r["same_c"]:.4f} '              f'gap_c={r["gap_c"]:.4f}')    del model, encoder, opt, tl, vl    if DEVICE == 'cuda':        torch.cuda.empty_cache()    gc.collect()    return rowsab = {}for _name, _mode, _lam in [('baseline', 'cosine', 0.0),                           ('raw',      'cosine', AB_LAMBDA),                           ('centered', 'centered', AB_LAMBDA)]:    print(f'\n--- {_name}: mode={_mode} lambda={_lam} ---')    ab[_name] = _ab_run(_mode, _lam)print(f'\n{"="*78}\nFINAL EPOCH\n{"="*78}')print(f'{"config":>10} {"val":>9} {"acc%":>7} {"sub_raw":>10} {"sub_cen":>10} '      f'{"same_c":>8} {"gap_c":>8}')for _k, _v in ab.items():    r = _v[-1]    print(f'{_k:>10} {r["val"]:>9.4f} {r["acc"]*100:>6.1f}% {r["raw"]:>10.6f} '          f'{r["cen"]:>10.6f} {r["same_c"]:>8.4f} {r["gap_c"]:>8.4f}')_b = ab['baseline'][-1]print('\nvs baseline (same-function cosine, centered):')for _k in ('raw', 'centered'):    r = ab[_k][-1]    print(f'  {_k:>8}: same_c {r["same_c"]-_b["same_c"]:+.4f} | '          f'val {r["val"]-_b["val"]:+.4f} | gap_c {r["gap_c"]-_b["gap_c"]:+.4f}')print(f'\nloss scale at final epoch: centered/raw = '      f'{_b["cen"]/max(_b["raw"], 1e-12):.1f}x')print('\nPick the mode that moves same_c without collapsing gap_c, then set')print('SUBSAMPLE_LOSS accordingly and re-run the config cell before sweeping.')

## SweepOnly run this once the smoke test looks right. 6 lambdas x 3 seeds = 18 runs.

In [ ]:
# ══════════════════════════════════════════════════════════════# Phase 1: train every run (no sympy in this phase)# ══════════════════════════════════════════════════════════════RUNS = [(l, s) for l in LAMBDA_VALUES for s in SEEDS]print(f'Phase 1: training {len(RUNS)} runs...')for lam, seed in RUNS:    train_one(lam, seed, synth_train, synth_val, tokenizer)# ══════════════════════════════════════════════════════════════# Phase 2: evaluate every run on the full test set# ══════════════════════════════════════════════════════════════print(f'\n\n{"="*70}')print(f'Phase 2: evaluating on the full test set ({len(synth_test)} eqs)...')print(f'{"="*70}')all_metrics = []for i, (lam, seed) in enumerate(RUNS):    run_tag = f'lam{lam}_seed{seed}'    t0 = time.time()    try:        m = eval_one(lam, seed, synth_val, synth_test, tokenizer)        print(f'  [{i+1}/{len(RUNS)}] {run_tag}: '              f'exact={m["greedy_exact_match"]*100:.1f}% | '              f'equiv={m["greedy_algebraic_equiv"]*100:.1f}% | '              f'gap_c={m["cos_gap_centered"]:.3f} | {time.time()-t0:.0f}s')        all_metrics.append(m)    except Exception as e:        print(f'  [{i+1}/{len(RUNS)}] {run_tag}: FAILED after '              f'{time.time()-t0:.0f}s — {e}')        all_metrics.append({            'lambda': lam, 'seed': seed, 'run_tag': run_tag,            'best_val_loss': float('nan'), 'best_val_acc': 0,            'greedy_exact_match': 0, 'greedy_algebraic_equiv': 0,            'greedy_r2_above_0.9': 0,            'same_fn_cos': float('nan'), 'diff_fn_cos': float('nan'),            'cos_gap': float('nan'), 'cos_gap_centered': float('nan'),            'same_fn_cos_centered': float('nan'),            'diff_fn_cos_centered': float('nan'),        })    gc.collect()

In [ ]:
# ── Per-run table ──print(f'{"="*104}')print(f'Subsample JEPA — {VERSION_TAG} | n_views={N_VIEWS} | n_test={len(synth_test)}')print(f'{"="*104}')print(f'\n{"lambda":>7} {"seed":>6} {"val_loss":>10} {"val_acc":>9} '      f'{"exact":>8} {"equiv":>8} {"R2>.9":>8} {"same_c":>9} '      f'{"diff_c":>9} {"gap_c":>9} {"gap_raw":>9}')print('-' * 104)for m in all_metrics:    print(f'{m["lambda"]:>7.3f} {m["seed"]:>6} {m["best_val_loss"]:>10.4f} '          f'{m.get("best_val_acc",0)*100:>8.1f}% '          f'{m["greedy_exact_match"]*100:>7.1f}% '          f'{m["greedy_algebraic_equiv"]*100:>7.1f}% '          f'{m["greedy_r2_above_0.9"]*100:>7.1f}% '          f'{m["same_fn_cos_centered"]:>9.4f} '          f'{m["diff_fn_cos_centered"]:>9.4f} '          f'{m["cos_gap_centered"]:>9.4f} {m["cos_gap"]:>9.4f}')# ── Mean across seeds ──print(f'\n{"lambda":>7} {"val_acc":>9} {"exact":>8} {"equiv":>8} {"R2>.9":>8} '      f'{"same_c":>9} {"diff_c":>9} {"gap_c":>9} {"gap_raw":>9} {"n":>4}')print('-' * 88)mean = lambda rs, k: float(np.mean([r[k] for r in rs]))for lam in LAMBDA_VALUES:    rs = [m for m in all_metrics if m['lambda'] == lam]    if not rs:        continue    print(f'{lam:>7.3f} {mean(rs,"best_val_acc")*100:>8.1f}% '          f'{mean(rs,"greedy_exact_match")*100:>7.1f}% '          f'{mean(rs,"greedy_algebraic_equiv")*100:>7.1f}% '          f'{mean(rs,"greedy_r2_above_0.9")*100:>7.1f}% '          f'{mean(rs,"same_fn_cos_centered"):>9.4f} '          f'{mean(rs,"diff_fn_cos_centered"):>9.4f} '          f'{mean(rs,"cos_gap_centered"):>9.4f} '          f'{mean(rs,"cos_gap"):>9.4f} {len(rs):>4}')

## Plots

In [ ]:
import matplotlib.pyplot as plt# Representation consistency and recovery vs lambda (mean +- range over seeds)def _agg(key):    mu, lo, hi = [], [], []    for lam in LAMBDA_VALUES:        vals = [m[key] for m in all_metrics if m['lambda'] == lam]        vals = [v for v in vals if v is not None and np.isfinite(v)]        if not vals:            vals = [np.nan]        mu.append(np.mean(vals)); lo.append(np.min(vals)); hi.append(np.max(vals))    mu, lo, hi = map(np.array, (mu, lo, hi))    return mu, mu - lo, hi - mux = np.arange(len(LAMBDA_VALUES))labels = [str(l) for l in LAMBDA_VALUES]fig, axes = plt.subplots(1, 3, figsize=(15, 4))for key, lab in [('same_fn_cos_centered', 'same-function'),                 ('diff_fn_cos_centered', 'different-function')]:    mu, el, eh = _agg(key)    axes[0].errorbar(x, mu, yerr=[el, eh], marker='o', capsize=3, label=lab)axes[0].set_title('Encoder cosine (mean-centered)'); axes[0].legend(fontsize=8)for key, lab, c in [('cos_gap_centered', 'centered', 'tab:green'),                    ('cos_gap', 'raw', 'tab:gray')]:    mu, el, eh = _agg(key)    axes[1].errorbar(x, mu, yerr=[el, eh], marker='o', capsize=3,                     color=c, label=lab)axes[1].set_title('same - different (headline metric)'); axes[1].legend(fontsize=8)for key, lab in [('greedy_algebraic_equiv', 'equiv'),                 ('greedy_exact_match', 'exact'),                 ('greedy_r2_above_0.9', 'R2>0.9')]:    mu, el, eh = _agg(key)    axes[2].errorbar(x, mu * 100, yerr=[el * 100, eh * 100],                     marker='o', capsize=3, label=lab)axes[2].set_title('Symbolic recovery (%)'); axes[2].legend(fontsize=8)for ax in axes:    ax.set_xticks(x); ax.set_xticklabels(labels)    ax.set_xlabel('lambda'); ax.grid(alpha=0.3)plt.tight_layout(); plt.show()

In [ ]:
# Training curves per runfig, axes = plt.subplots(1, 4, figsize=(20, 4))for lam in LAMBDA_VALUES:    for seed in SEEDS:        p = f'{CKPT_DIR}/{VERSION_TAG}/lam{lam}_seed{seed}/latest.pt'        if not os.path.exists(p):            continue        h = torch.load(p, map_location='cpu', weights_only=False)['history']        label = f'l={lam} s={seed}'        a = 0.5 if len(SEEDS) > 1 else 1.0        axes[0].plot(h['train'], label=label, alpha=a)        axes[1].plot(h['val'], label=label, alpha=a)        if h.get('val_acc'):            axes[2].plot([v * 100 for v in h['val_acc']], label=label, alpha=a)        if h.get('val_gap_centered'):            axes[3].plot(h['val_gap_centered'], label=label, alpha=a)for ax, t in zip(axes, ['Train CE', 'Val CE', 'Val token acc %',                        'same - diff cosine (centered)']):    ax.set_title(t); ax.set_xlabel('epoch'); ax.grid(alpha=0.3)    ax.legend(fontsize=6)plt.tight_layout(); plt.show()

In [ ]:
%load_ext tensorboard%tensorboard --logdir {LOG_DIR}

## Inspect predictions

In [ ]:
def to_infix(prefix_str):    """Prefix string -> infix via SymPy. Returns (infix, error)."""    try:        expr, _ = prefix_to_sympy(prefix_str)        return str(expr), None    except Exception as e:        return None, str(e)def inspect_run(run_tag, n_show=10, tag=VERSION_TAG):    path = f'{CKPT_DIR}/{tag}/{run_tag}/metrics.json'    if not os.path.exists(path):        print(f'{run_tag}: no metrics.json'); return    with open(path) as f:        m = json.load(f)    details = m.get('details', [])    print(f'\n{"="*80}')    print(f'{run_tag} | exact={m["greedy_exact_match"]*100:.1f}% | '          f'equiv={m["greedy_algebraic_equiv"]*100:.1f}% | '          f'gap_c={m["cos_gap_centered"]:.3f}')    print(f'{"="*80}')    for i, d in enumerate(details[:n_show]):        gt, _ = to_infix(d['gt'])        pred, err = to_infix(d['pred'])        r2 = f'{d["r2"]:.4f}' if d.get('r2') is not None else 'N/A'        status = ('EXACT' if d['exact'] else 'EQUIV' if d.get('equiv')                  else 'PARSEABLE' if d.get('parseable') else 'UNPARSEABLE')        print(f'\n-- [{i}] {status} | R2={r2}')        print(f'  GT:   {gt}')        print(f'  Pred: {pred if err is None else "PARSE FAILED: " + err}')for lam in LAMBDA_VALUES:    inspect_run(f'lam{lam}_seed{SEEDS[0]}', n_show=5)